# 02. Exploratory Data Analysis

This notebook performs a concise exploratory data analysis (EDA) of the merged preprocessed news corpus before vectorisation.

## Purpose
The aim of this notebook is to:
- summarise the size and composition of the corpus
- examine article length patterns across sources
- confirm data quality after preprocessing
- inspect the vocabulary size of the merged corpus
- generate a word cloud of the most frequent terms

## Expected input
This notebook expects the cleaned merged dataset produced in the preprocessing stage:

```python
combined_clean_deduplicated.xlsx
```

## Expected output
The notebook produces:
- a dataset summary table by source
- corpus-level summary statistics
- an article length distribution plot
- a source distribution plot
- a word cloud of frequent terms


## 1. Import libraries

This section imports the required libraries for data analysis and visualisation.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud

## 2. Define input file

**Note:** The dataset is not included in the public repository due to size and data availability constraints.


In [ ]:
input_file = "combined_clean_deduplicated.xlsx" 

## 3. Load the dataset

In [ ]:
df = pd.read_excel(input_file)
print("Dataset shape:", df.shape)
df.head()

## 4. Basic data quality checks

This section confirms whether there are any missing or empty records remaining after preprocessing.


In [ ]:
missing_clean_text = df["clean_text"].isna().sum()
blank_clean_text = (df["clean_text"].astype(str).str.strip() == "").sum()
duplicate_clean_text = df.duplicated(subset="clean_text").sum()

print("Missing clean_text values:", missing_clean_text)
print("Blank clean_text values:", blank_clean_text)
print("Duplicate clean_text values:", duplicate_clean_text)

## 5. Compute article length statistics

Article length is measured as the number of words in each cleaned article.


In [ ]:
df["article_length_words"] = df["clean_text"].astype(str).str.split().apply(len)

print("Overall average article length:", round(df["article_length_words"].mean(), 2))
print("Median article length:", round(df["article_length_words"].median(), 2))

## 6. Source-level summary table

This table reports the number of articles and average article length for each source.


In [ ]:
summary_table = (
    df.groupby("source")
      .agg(
          Number_of_Articles=("clean_text", "size"),
          Average_Article_Length_Words=("article_length_words", "mean")
      )
      .reset_index()
)

summary_table["Average_Article_Length_Words"] = summary_table["Average_Article_Length_Words"].round(2)
summary_table = summary_table.sort_values("Number_of_Articles", ascending=False).reset_index(drop=True)

total_row = pd.DataFrame({
    "source": ["Total"],
    "Number_of_Articles": [len(df)],
    "Average_Article_Length_Words": [round(df["article_length_words"].mean(), 2)]
})

summary_table_final = pd.concat([summary_table, total_row], ignore_index=True)
summary_table_final

## 7. Vocabulary size

Vocabulary size is calculated as the number of unique tokens in the merged corpus.


In [ ]:
all_tokens = " ".join(df["clean_text"].astype(str)).split()
vocab_size = len(set(all_tokens))

print("Vocabulary size:", vocab_size)

## 8. Source distribution plot

This plot shows the number of articles contributed by each source.


In [ ]:
source_counts = df["source"].value_counts()

plt.figure(figsize=(8, 5))
source_counts.plot(kind="bar")
plt.title("Number of Articles by Source")
plt.xlabel("Source")
plt.ylabel("Number of Articles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Article length distribution

The distribution is expected to be right-skewed because the corpus contains many short AG News records alongside longer Reuters, BBC, and scraped articles.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["article_length_words"], bins=50)
plt.title("Distribution of Article Lengths")
plt.xlabel("Article Length (words)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 10. Word cloud of frequent terms

The word cloud provides a quick visual overview of dominant terms in the corpus after preprocessing.


In [ ]:
word_frequencies = Counter(all_tokens)

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    max_words=200
).generate_from_frequencies(word_frequencies)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Frequent Terms")
plt.tight_layout()
plt.show()

## 11. Observations

The merged corpus contains a large number of articles drawn from four different sources, providing a diverse basis for downstream clustering. The summary table highlights a clear source imbalance, with AG News contributing the majority of records, while Reuters, BBC, and the scraped set contribute fewer but generally longer articles.

Article length distribution is right-skewed, which reflects the real-world variation between short summaries and longer news reports. This makes the dataset more realistic and also more challenging for clustering.

The word cloud highlights broadly news-oriented vocabulary such as terms related to time, companies, and current events. The absence of trivial stopwords provides a useful sanity check that preprocessing was effective.

Overall, the EDA confirms that the corpus is large, varied, and suitable for vectorisation and clustering experiments.
